# Assignment: K-Fold Cross-Validation

**Topic:** Model Evaluation — Train/Test Split vs. K-Fold Cross-Validation

### Learning Objectives
By the end of this assignment, you should be able to:
1. Explain why a single train/test split can give a misleading estimate of model performance.
2. Implement K-Fold Cross-Validation using scikit-learn.
3. Use `cross_val_score` to get per-fold scores, and compute their mean and standard deviation.
4. Compare results across different values of `k`.
5. Explain, in your own words, what cross-validation protects you against.

### Dataset
This assignment uses the classic **Iris dataset** (built into scikit-learn — no download
needed). It has 150 flower samples, 4 numeric features (petal/sepal length & width), and
a target `species` with 3 classes. It's small and simple on purpose, so you can focus
entirely on the cross-validation concepts rather than data cleaning.

Run the cells in order. Cells marked **`# TODO`** are for you to complete. Markdown cells
with **`Q:`** ask conceptual questions — answer them directly underneath.

## Part 0 — Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

## Part 1 — Load the Data

**TODO:** Load the Iris dataset using `sklearn.datasets.load_iris`, and store the
features in `X` and the target labels in `y`.

In [2]:
from sklearn.datasets import load_iris

# TODO: load the iris dataset
iris = load_iris()

# TODO: store features in X and target in y
X = iris.data

y = iris.target

# TODO: print the shape of X and the number of unique classes in y
print("X shape:", X.shape)
print("Number of classes:", len(np.unique(y)))

X shape: (150, 4)
Number of classes: 3


## Part 2 — Baseline: A Single Train/Test Split

**TODO:**
1. Split `X, y` into `train_X, test_X, train_y, test_y` using `train_test_split`
   with `test_size=0.3` and `random_state=1`.
2. Fit a `DecisionTreeClassifier()` on the training data.
3. Print its accuracy (`.score()`) on the test set.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# TODO: split the data
train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.3, random_state=1)

# TODO: fit a DecisionTreeClassifier
model = DecisionTreeClassifier(random_state=0)
model.fit(train_X, train_y)

# TODO: Print the accuracy
print("Test accuracy:", model.score(test_X, test_y))


Test accuracy: 0.9555555555555556


## Part 3 — The Problem: Accuracy Depends on the Split

**TODO:** Repeat Part 2, but this time loop over **5 different `random_state` values**
(e.g. `0, 1, 2, 3, 4`) for the train/test split. For each one, fit a fresh
`DecisionTreeClassifier` and print the test accuracy. Store all 5 accuracies in a list.

In [5]:
# TODO: loop over 5 random_state values, split, fit, and record test accuracy each time
accuracies = []

for state in [0, 1, 2, 3, 4]:
    train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.3, random_state=state)
    model = DecisionTreeClassifier(random_state=0)
    model.fit(train_X, train_y)
    accuracies.append(model.score(test_X, test_y))
    
print(accuracies)
print("Range of accuracies:", min(accuracies), "to", max(accuracies))

[0.9777777777777777, 0.9555555555555556, 0.9555555555555556, 0.9111111111111111, 0.9777777777777777]
Range of accuracies: 0.9111111111111111 to 0.9777777777777777


**Q1:** How much did the test accuracy vary just by changing which rows ended up in
the test set (with the *same* model and the *same* amount of data)? Why is this a
problem if you only ever look at one train/test split?

*Your answer:*
The range of accuracies is between 0.9111 to 0.911111, a roughly 6-7 percentage point swing — even though the model and the amount of data are identical each time; the only thing that changed is which 45 flowers happened to land in the test set. If you only ever looked at one split, you could easily conclude the model is better or worse than it actually is, purely due to that split's luck. With a small dataset like Iris (150 rows), a handful of "hard" or "easy" samples landing in the test set can noticeably swing accuracy.

## Part 4 — K-Fold Cross-Validation

Instead of one split, **K-Fold Cross-Validation** divides the data into `k` equal-sized
folds. It trains on `k-1` folds and tests on the remaining fold, then repeats this `k`
times so that every fold gets a turn as the test set. This gives you `k` accuracy scores
instead of just one.

**TODO:**
1. Create a `KFold` object with `n_splits=5, shuffle=True, random_state=1`.
2. Use `cross_val_score` to evaluate a fresh `DecisionTreeClassifier()` across the 5 folds.
3. Print the 5 individual fold scores, then print their mean and standard deviation.

In [6]:
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=1)

scores = cross_val_score(DecisionTreeClassifier(random_state=0), X, y, cv=kf)

print("Fold scores:", scores)
print("Mean accuracy:", scores.mean())
print("Std deviation:", scores.std())

Fold scores: [0.96666667 0.96666667 0.96666667 0.93333333 0.83333333]
Mean accuracy: 0.9333333333333332
Std deviation: 0.05163977794943222


**Q2:** Compare the mean of your 5 K-Fold scores to the range of accuracies you saw
in Part 3. Is the K-Fold mean a more trustworthy estimate of how the model will perform
on new data? Why or why not?

*Your answer:*
Yes — the K-Fold mean (0.933, std ≈ 0.052) is a more trustworthy estimate than any single split. Instead of relying on one lucky-or-unlucky split, it averages performance over 5 different train/validation partitions, so every row gets used for validation exactly once. This smooths out the split-to-split noise we saw in Part 3 — notice fold 5 alone scored only 0.833, well below the Part 2/3 single-split results, but because it's averaged with four other folds (0.967, 0.967, 0.967, 0.933) the overall mean stays representative rather than being thrown off by that one weaker fold.

## Part 5 — Trying Different Values of k

**TODO:** Repeat the K-Fold cross-validation from Part 4, but try `k = 3`, `k = 5`, and
`k = 10`. For each value of `k`, print the mean accuracy and the standard deviation of
the fold scores.

In [7]:
# TODO: loop over k = 3, 5, 10 and print mean + std of cross_val_score for each
for k in [3, 5, 10]:
    kf_k = KFold(n_splits=k, shuffle=True, random_state=1)
    scores_k = cross_val_score(DecisionTreeClassifier(random_state=0), X, y, cv=kf_k)
    print(f"k={k:>2}  mean={scores_k.mean():.4f}  std={scores_k.std():.4f}  scores={np.round(scores_k, 3)}")

k= 3  mean=0.9200  std=0.0283  scores=[0.96 0.9  0.9 ]
k= 5  mean=0.9333  std=0.0516  scores=[0.967 0.967 0.967 0.933 0.833]
k=10  mean=0.9400  std=0.0629  scores=[1.    0.933 0.933 1.    1.    0.933 1.    0.867 0.933 0.8  ]


**Q3:** What happens to the standard deviation of the fold scores as `k` increases?
What trade-off do you think there is between using a small `k` (like 3) versus a large
`k` (like 10)? (Hint: think about how much data is used for training vs. testing in each
fold, and how long cross-validation takes to run.)

*Your answer:*
In this run: k=3 → mean 0.920, std 0.028; k=5 → mean 0.933, std 0.052; k=10 → mean 0.940, std 0.063. As k increases, each fold's validation portion shrinks (33% held out per fold at k=3, down to just 10% at k=10) while each fold's training portion grows — with k=10, the model sees ~90% of the data each time, so the mean accuracy creeps up slightly. But with only ~15 samples per validation fold at k=10, a single misclassified flower changes that fold's score by a large chunk (e.g. one miss out of 15 ≈ a 6-7% swing), which is exactly why the standard deviation is highest at k=10 in these results, not lowest. The trade-off: small k trains on less data per fold (slightly more pessimistic/biased mean) but each fold's score is smoother (based on more validation samples); large k trains on almost all the data each time (less biased mean) but each fold's score is noisier and cross-validation takes longer to run (more model fits). k=5 or k=10 are common defaults that balance this reasonably well.

## Part 6 — Stratified K-Fold (Bonus, Optional)

The Iris dataset has 3 balanced classes (50 samples each). `StratifiedKFold` makes sure
each fold has roughly the same proportion of each class — important for classification,
especially with imbalanced classes.

**TODO (optional):** Use `StratifiedKFold(n_splits=5)` with `cross_val_score` and compare
the mean accuracy to your plain `KFold` result from Part 4.

In [8]:
# Optional bonus code
from sklearn.model_selection import StratifiedKFold

# TODO: try StratifiedKFold and compare to KFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
strat_scores = cross_val_score(DecisionTreeClassifier(random_state=0), X, y, cv=skf)

print("Stratified fold scores:", strat_scores)
print("Stratified mean accuracy:", strat_scores.mean())
print()
print("Plain KFold mean accuracy:   ", scores.mean())
print("Stratified KFold mean accuracy:", strat_scores.mean())

Stratified fold scores: [0.96666667 1.         0.9        1.         0.86666667]
Stratified mean accuracy: 0.9466666666666667

Plain KFold mean accuracy:    0.9333333333333332
Stratified KFold mean accuracy: 0.9466666666666667


## Part 7 — Reflection

**Q4:** In your own words, what does K-Fold Cross-Validation protect you against that a
single train/test split does not?

*Your answer:*
A single train/test split gives you exactly one performance number, and that number depends heavily on which specific rows ended up in the test set — it can make an average model look great or a good model look mediocre, purely by chance. K-Fold Cross-Validation protects against this by evaluating the model on several different train/validation partitions and averaging the results, giving an estimate that reflects the model's typical performance rather than its performance on one particular random split. It also uses every row for validation at some point, rather than "wasting" a chunk of data purely as a held-out test set.

**Q5:** If you were choosing between two models and one had a mean CV accuracy of 0.94
with a standard deviation of 0.01, and the other had a mean CV accuracy of 0.95 with a
standard deviation of 0.08, which model would you trust more for making predictions on
new data? Explain your reasoning.

*Your answer:*
The first model (mean 0.94, std 0.01) is the more trustworthy choice for predictions on new data, even though its mean is slightly lower than the second model's (0.95). A low standard deviation means the model performs consistently across different subsets of data — you can be fairly confident it will score close to 0.94 on new, unseen data. A high standard deviation (0.08) means the second model's performance swings a lot depending on which data it sees — on some data it might score close to 1.0, but on other data it could drop well below 0.90. When comparing two models with similar means, prefer the one with the tighter (lower-variance) spread of fold scores.
